---
# Import libraries and data

In [1]:
# base modules
from pathlib import Path
import logging
from collections import OrderedDict

# for manipulating data
import numpy as np
import pandas as pd
import math
from typing import Callable
import copy
import re
from pandas.api.types import is_string_dtype, is_numeric_dtype


# for Machine Learning
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn import metrics
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV,train_test_split

# for visualization
from matplotlib import pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# for package auto reload
%load_ext autoreload
%autoreload 2

# for better rendering of plots in jupyter notebook
%matplotlib inline

In [2]:
import optuna

In [3]:
import sys
import os
sys.path.append(os.path.abspath("../src"))
from process_data import log_transform_targets, process_df_keep_nan, clip_outliers

In [4]:
df=pd.read_csv("../data/merged_training_data_clean.csv")

---
# Data Cleaning

In [5]:
df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,blue,green,red,nir08,swir16,swir22,NDMI,MNDWI,pet,ppt,tmax,tmin,q
0,-28.760833,17.730278,2011-01-02,128.912,555.0,10.0,9881.0,11481.0,12405.0,11761.5,7791.0,7746.0,0.185538,0.195595,174.2,8.3,35.760000,21.029999,0.4
1,-26.861111,28.884722,2011-01-03,74.720,162.9,163.0,NaN,9550.0,NaN,17658.5,13746.5,10574.0,0.124566,-0.180134,124.1,175.0,25.410000,14.099999,8.8
2,-26.450000,28.085833,2011-01-03,89.254,573.0,80.0,10246.0,10824.0,10213.5,19147.0,15023.5,11040.0,-0.083293,-0.252805,127.5,156.1,26.070000,15.250000,7.8
3,-27.671111,27.236944,2011-01-03,82.000,203.6,101.0,NaN,10943.0,NaN,14887.0,13522.0,11403.0,0.048048,-0.105416,129.7,194.7,27.599998,15.960000,16.1
4,-27.356667,27.286389,2011-01-03,56.100,145.1,151.0,NaN,9502.5,NaN,16828.5,12665.5,9643.0,0.141147,-0.142683,129.2,180.4,27.210000,16.150000,9.0


In [6]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Latitude,9319.0,NaN,NaN,NaN,-28.474988,2.760282,-34.405833,-30.160091,-28.058889,-26.861111,-22.225556
Longitude,9319.0,NaN,NaN,NaN,26.868414,3.535164,17.730278,26.126667,27.40906,29.245556,32.325
Sample Date,9319,1364,2014-01-08,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Total Alkalinity,9319.0,NaN,NaN,NaN,119.108208,74.692591,4.8,55.811,113.3,170.23,361.676
Electrical Conductance,9319.0,NaN,NaN,NaN,485.004146,341.937736,15.12,207.05,402.0,693.0,1506.0
Dissolved Reactive Phosphorus,9319.0,NaN,NaN,NaN,43.525338,50.980194,5.0,10.0,20.0,48.0,195.0
blue,8178.0,NaN,NaN,NaN,9049.943752,1826.051769,6071.0,8543.0,8924.0,9323.0,54475.5
green,9078.0,NaN,NaN,NaN,10013.609385,1855.255857,4438.5,9429.25,9848.25,10338.25,52339.0
red,8178.0,NaN,NaN,NaN,10336.573245,1938.733531,7179.0,9460.5,10129.0,10951.0,51854.5
nir08,9078.0,NaN,NaN,NaN,14217.92592,2855.227789,4884.0,12826.0,14311.5,15635.625,49593.0


In [7]:
target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

In [8]:
df['Sample Date'] = pd.to_datetime(df['Sample Date'])
df['year']   = df['Sample Date'].dt.year
df['month']  = df['Sample Date'].dt.month
df['day']    = df['Sample Date'].dt.day
df['doy']    = df['Sample Date'].dt.dayofyear
df['season'] = df['month'] % 12 // 3

In [9]:
df=log_transform_targets(df, target_cols)

In [10]:
for col in ['q', 'ppt', 'tmax', 'tmin']:
    df = clip_outliers(df, col, 0.01, 0.99)

In [12]:
df_pr, y = process_df_keep_nan(
   df,
    y_fields=target_cols,
    skip_flds=['Sample Date']
)

In [13]:
df_pr.shape, y.shape

((9319, 28), (9319, 3))

In [15]:
df_pr.columns

Index(['Latitude', 'Longitude', 'blue', 'green', 'red', 'nir08', 'swir16',
       'swir22', 'NDMI', 'MNDWI', 'pet', 'ppt', 'tmax', 'tmin', 'q', 'year',
       'month', 'day', 'doy', 'season', 'blue_na', 'green_na', 'red_na',
       'nir08_na', 'swir16_na', 'swir22_na', 'NDMI_na', 'MNDWI_na'],
      dtype='object')

--- 
# Model

In [16]:
N_valid = 1000
N_small = 2000
X_train, X_valid, y_train, y_valid = train_test_split(df_pr, y, test_size=N_valid, random_state=42)
_, X_small, _, y_small = train_test_split(X_train, y_train, test_size=N_small, random_state=42)
print("Large training set size : ",X_train.shape, y_train.shape)
print("Small training set size : ",X_small.shape, y_small.shape)
print("Validation set size : ",X_valid.shape, y_valid.shape)

Large training set size :  (8319, 28) (8319, 3)
Small training set size :  (2000, 28) (2000, 3)
Validation set size :  (1000, 28) (1000, 3)


In [17]:
def rmse(y_gold, y_pred):
    """
    y_gold, y_pred: 可以是一维或二维（多输出）
    返回：整体 RMSE（对所有样本 & 所有 target 统一算一个）
    """
    y_gold = np.asarray(y_gold)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_gold - y_pred) ** 2)))

In [18]:
def print_score(m, X_train, y_train, X_valid, y_valid, log_target=True):
    y_train_pred_log = m.predict(X_train)
    y_valid_pred_log = m.predict(X_valid)
    print('1. RMSE on train set (log_y): {:.4f}'.format(rmse(y_train, y_train_pred_log)))
    print('2. RMSE on valid set (log_y): {:.4f}'.format(rmse(y_valid, y_valid_pred_log)))
    print('3. MAE on train set (log_y): {:.4f}'.format(mean_absolute_error(y_train, y_train_pred_log)))
    print('4. MAE on valid set (log_y): {:.4f}'.format(mean_absolute_error(y_valid, y_valid_pred_log)))
    if log_target:
        y_train_real      = np.expm1(y_train)
        y_valid_real      = np.expm1(y_valid)
        y_train_pred_real = np.expm1(y_train_pred_log)
        y_valid_pred_real = np.expm1(y_valid_pred_log)
        print('5. R^2 on train set (original y): {:.4f}'.format(
            r2_score(y_train_real, y_train_pred_real)))
        print('6. R^2 on valid set (original y): {:.4f}'.format(
            r2_score(y_valid_real, y_valid_pred_real)))

    else:
        print('5. R^2 on train set: {:.4f}'.format(m.score(X_train, y_train)))
        print('6. R^2 on valid set: {:.4f}'.format(m.score(X_valid, y_valid)))
    if hasattr(m, 'oob_score_'):
        print('R^2 on oob set: {:.4f}'.format(m.oob_score_))

    return

--- 
# Random Forest

In [19]:
def objective(trial, X_training, y_training, X_validation, y_validation):
    n_estimators = trial.suggest_int('n_estimators', 100, 2500) 
    max_depth = trial.suggest_int('max_depth', 5, 60, log=True) 
    min_samples_split = trial.suggest_int('min_samples_split', 2, 50, log=True) 
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 30, log=True) 
    max_features = trial.suggest_float('max_features', 0.1, 1.0)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])
    max_samples = trial.suggest_float('max_samples', 0.5, 1.0, step=0.05) if bootstrap else None
    
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        max_samples=max_samples, 
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X_training, y_training)

    y_pred_validation = rf.predict(X_validation)
    r2_validation = metrics.r2_score(y_validation, y_pred_validation)

    return r2_validation

In [20]:
study = optuna.create_study(
    study_name="rf_study2", 
    load_if_exists=True, 
    storage="sqlite:///opt.db",
    direction="maximize"
)
study.optimize(
    lambda trial: objective(trial, X_small, y_small, X_valid, y_valid),
    n_trials=50,
    show_progress_bar=True
)

[I 2026-03-03 10:50:38,813] A new study created in RDB with name: rf_study2


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-03 10:50:41,620] Trial 0 finished with value: 0.7398718799462145 and parameters: {'n_estimators': 1559, 'max_depth': 45, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.532900090296172, 'bootstrap': False}. Best is trial 0 with value: 0.7398718799462145.
[I 2026-03-03 10:50:42,144] Trial 1 finished with value: 0.5758201235219644 and parameters: {'n_estimators': 634, 'max_depth': 13, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 0.2090641769247762, 'bootstrap': False}. Best is trial 0 with value: 0.7398718799462145.
[I 2026-03-03 10:50:44,022] Trial 2 finished with value: 0.4899979274335054 and parameters: {'n_estimators': 2458, 'max_depth': 5, 'min_samples_split': 3, 'min_samples_leaf': 13, 'max_features': 0.4970017277865053, 'bootstrap': False}. Best is trial 0 with value: 0.7398718799462145.
[I 2026-03-03 10:50:45,456] Trial 3 finished with value: 0.6513378941317579 and parameters: {'n_estimators': 1262, 'max_depth': 38, 'min_samples_spli

In [21]:
study.best_params

{'n_estimators': 1751,
 'max_depth': 38,
 'min_samples_split': 4,
 'min_samples_leaf': 2,
 'max_features': 0.658035356275404,
 'bootstrap': False}

In [22]:
rf_best = RandomForestRegressor(
        n_estimators=1751,
        max_depth=38,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features=0.658035356275404,
        bootstrap=False,
        random_state=42,
        n_jobs=-1,
    )
%time rf_best.fit(X_train, y_train)
print_score(rf_best, X_train, y_train, X_valid, y_valid)

CPU times: total: 2min 34s
Wall time: 11.8 s
1. RMSE on train set (log_y): 0.1058
2. RMSE on valid set (log_y): 0.3649
3. MAE on train set (log_y): 0.0606
4. MAE on valid set (log_y): 0.2237
5. R^2 on train set (original y): 0.9762
6. R^2 on valid set (original y): 0.8145


In [23]:
import joblib

joblib.dump(rf_best, '../user_data/rf_best.joblib')

['../user_data/rf_best.joblib']

---
# XGBoost

In [24]:
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor, Pool, cv, CatBoostClassifier
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV, train_test_split
from sklearn import metrics
from xgboost import XGBRegressor, plot_tree
import seaborn as sns

In [25]:
def reg_perf(regressor, X_training, X_validation, y_true_training, y_true_validation):
    y_pred_training = regressor.predict(X_training)
    y_pred_validation = regressor.predict(X_validation)
    rmse_training = metrics.root_mean_squared_error(y_true_training, y_pred_training)
    rmse_validation = metrics.root_mean_squared_error(y_true_validation, y_pred_validation)
    r2_training = metrics.r2_score(y_true_training, y_pred_training)
    r2_validation = metrics.r2_score(y_true_validation, y_pred_validation)
    print("RMSE on training set : {:.3f}".format(rmse_training))
    print("RMSE on validation set : {:.3f}".format(rmse_validation))
    print("R^2 on training set : {:.3f}".format(r2_training))
    print("R^2 on validation set : {:.3f}".format(r2_validation))

In [26]:
def objective(trial, X_training, y_training, X_validation, y_validation):
    
    learning_rate_guess = trial.suggest_float("learning_rate", 1e-3, 1, log=True)
    max_depth_guess = trial.suggest_int("max_depth", 2, 30, log=False)
    reg_lambda_guess = trial.suggest_int("reg_lambda", 1,100, log=False)
    colsample_bytree_guess = trial.suggest_float("colsample_bytree", 0.3, 1.0, log=False)
    min_child_weight_guess = trial.suggest_int("min_child_weight", 1, 10, log=False)
    subsample_guess = trial.suggest_float("subsample", 0.5, 1.0, log=False)
    gamma_guess = trial.suggest_float("gamma", 0.0, 5.0, log=False)
    reg_alpha_guess = trial.suggest_float("reg_alpha", 1e-8, 100.0, log=True)

    
    xgb = XGBRegressor(
        n_estimators=1000,                              
        random_state=42,
        learning_rate=learning_rate_guess,       
        max_depth=max_depth_guess, 
        reg_lambda=reg_lambda_guess,
        verbosity=0,
        colsample_bytree=colsample_bytree_guess,
        min_child_weight=min_child_weight_guess,   
        subsample=subsample_guess  ,
        gamma=gamma_guess,      
        reg_alpha=reg_alpha_guess  
    )
    
    xgb.fit(X_training, y_training)
    
    y_pred_validation = xgb.predict(X_validation)
    r2_validation = metrics.r2_score(y_validation, y_pred_validation)
    return r2_validation

In [27]:
study = optuna.create_study(study_name="xgb_study2", 
                            load_if_exists=True, 
                            storage = "sqlite:///opt.db",
                            direction="maximize")

study.optimize(lambda trial: objective(trial, X_small, y_small, X_valid, y_valid), n_trials=50, show_progress_bar=True)

[I 2026-03-03 10:54:07,690] A new study created in RDB with name: xgb_study2


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-03 10:54:10,711] Trial 0 finished with value: 0.7034945973458772 and parameters: {'learning_rate': 0.1068831825935791, 'max_depth': 11, 'reg_lambda': 15, 'colsample_bytree': 0.7074345386101175, 'min_child_weight': 4, 'subsample': 0.9137509426967863, 'gamma': 0.7964549120981745, 'reg_alpha': 9.098022120788824e-07}. Best is trial 0 with value: 0.7034945973458772.
[I 2026-03-03 10:54:11,365] Trial 1 finished with value: 0.6469519357807345 and parameters: {'learning_rate': 0.057311557033451185, 'max_depth': 24, 'reg_lambda': 38, 'colsample_bytree': 0.596319784546688, 'min_child_weight': 8, 'subsample': 0.5193759819921581, 'gamma': 1.0641643423362024, 'reg_alpha': 7.529497617260533e-05}. Best is trial 0 with value: 0.7034945973458772.
[I 2026-03-03 10:54:11,834] Trial 2 finished with value: 0.6685444992616735 and parameters: {'learning_rate': 0.48235698395665755, 'max_depth': 27, 'reg_lambda': 19, 'colsample_bytree': 0.32549409567726245, 'min_child_weight': 5, 'subsample': 0.8247

In [28]:
study.best_params

{'learning_rate': 0.02212488542871817,
 'max_depth': 30,
 'reg_lambda': 5,
 'colsample_bytree': 0.915830103274166,
 'min_child_weight': 2,
 'subsample': 0.6994086343520101,
 'gamma': 0.40877113364972884,
 'reg_alpha': 0.00037891042863194194}

In [29]:
xgb_best = XGBRegressor(
        n_estimators=1000,                              
        random_state=42,
        learning_rate= 0.02212488542871817,       
        max_depth=30, 
        reg_lambda=5,
        verbosity=0,
        colsample_bytree= 0.915830103274166,
        min_child_weight=2,   
        subsample=0.6994086343520101 ,
        gamma=0.40877113364972884 ,
        reg_alpha   =0.00037891042863194194
)
%time xgb_best.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])
reg_perf(xgb_best, X_train, X_valid, y_train, y_valid)

[0]	validation_0-rmse:0.86774
[1]	validation_0-rmse:0.85560
[2]	validation_0-rmse:0.84291
[3]	validation_0-rmse:0.83019
[4]	validation_0-rmse:0.81731
[5]	validation_0-rmse:0.80494
[6]	validation_0-rmse:0.79290
[7]	validation_0-rmse:0.78164
[8]	validation_0-rmse:0.77053
[9]	validation_0-rmse:0.75930
[10]	validation_0-rmse:0.74843
[11]	validation_0-rmse:0.73868
[12]	validation_0-rmse:0.72839
[13]	validation_0-rmse:0.71834
[14]	validation_0-rmse:0.70856
[15]	validation_0-rmse:0.69950
[16]	validation_0-rmse:0.69004
[17]	validation_0-rmse:0.68128
[18]	validation_0-rmse:0.67244
[19]	validation_0-rmse:0.66375
[20]	validation_0-rmse:0.65578
[21]	validation_0-rmse:0.64761
[22]	validation_0-rmse:0.63987
[23]	validation_0-rmse:0.63228
[24]	validation_0-rmse:0.62477
[25]	validation_0-rmse:0.61766
[26]	validation_0-rmse:0.61042
[27]	validation_0-rmse:0.60367
[28]	validation_0-rmse:0.59709
[29]	validation_0-rmse:0.59041
[30]	validation_0-rmse:0.58470
[31]	validation_0-rmse:0.57887
[32]	validation_0-

In [30]:
joblib.dump(xgb_best, '../user_data/XGBoost_Model.joblib')

['../user_data/XGBoost_Model.joblib']

---
# Submission


In [31]:
val=pd.read_csv("../data/merged_validation_data_clean.csv")

In [32]:
val["Sample Date"] = pd.to_datetime(val["Sample Date"], errors="coerce")

val["year"] = val["Sample Date"].dt.year
val["month"] = val["Sample Date"].dt.month
val["doy"] = val["Sample Date"].dt.dayofyear
val["season"] = val["month"] % 12 // 3

X_test, _ = process_df_keep_nan(val, y_fields=None, skip_flds=["Sample Date"])
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_pred_log = xgb_best.predict(X_test)
y_pred = np.expm1(y_pred_log)

sub = pd.read_csv("../resources/data/submission_template.csv")
sub["Total Alkalinity"] = y_pred[:,0]
sub["Electrical Conductance"] = y_pred[:,1]
sub["Dissolved Reactive Phosphorus"] = y_pred[:,2]
sub.to_csv("../user_data/submission.csv", index=False)

In [33]:
pred = xgb_best.predict(X_test)
print(pred.shape)

(200, 3)


In [34]:
print("log pred example:", y_pred_log[0])
print("real pred example:", np.expm1(y_pred_log[0]))

log pred example: [2.776441  5.400292  3.0970461]
real pred example: [ 15.061756 220.47105   21.132479]


In [36]:
import pandas as pd
ls = pd.read_csv("../data/processed/landsat_api_validation.csv")
print(ls["Index"].min(), ls["Index"].max(), ls["Index"].nunique())

0 199 200


In [37]:
val = pd.read_csv("../data/merged_validation_data_clean.csv")
print(val[["blue","green","red","nir08","swir16","swir22","pet","ppt","tmax","tmin","q"]].describe().T)

        count          mean          std      min           25%          50%  \
blue    172.0   9027.183140  2408.712104  7770.00   8208.125000   8639.25000   
green   196.0   9955.094388  2285.693956  8210.50   9020.625000   9622.50000   
red     172.0  10203.186047  2544.293075  8065.50   9003.000000   9773.25000   
nir08   196.0  14937.841837  2828.830922  8823.50  13416.125000  14684.00000   
swir16  196.0  12562.920918  1928.079536  8307.00  11334.000000  12425.75000   
swir22  196.0  10419.066327  1547.392099  8107.50   9221.000000  10122.25000   
pet     200.0    160.278003    25.346462    60.50    140.750000    158.40001   
ppt     200.0     50.091500    40.372768     1.20     18.850000     38.55000   
tmax    200.0     24.422150     3.350726    15.48     22.289999     24.34000   
tmin    200.0     11.699300     3.956054    -0.85      9.300000     12.00500   
q       200.0      3.098500     5.743893     0.10      0.975000      1.95000   

              75%       max  
blue     